# Dataset Characterization

the idea of this notebook is to characterize the dataset only, regarding only structures dft data, maybe composition , ternary plot, and this kind of things.

input: `CuratedBriefSummary.pkl` 

output: plots

In [ ]:
from Tools.DatasetTools.Commoms import *
dataset = 'Fe-Mo'  # 'Cr-Co-W'#'Fe-Mo
target_case = 'EF_nmhcp'
components = dataset.split('-')
# sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer  import Featurizer
plt.style.use('default')
#plt.rc('figure', figsize=(15,10))
plt.rc('font', size=22)
plt.rc('text', usetex=True)


In [ ]:
BSfile = os.path.join(dataset,'CuratedParsedBriefSummary.pkl' )
BS = pd.read_pickle(BSfile)

In [ ]:
BS.shape

# Population of phases

In [ ]:
BS.drop('Fe_pv30.sigma_Fe_pv.FM', inplace=True)

In [ ]:
phase_counts = pd.concat([BS['Phase'][BS['Mag']=='NM'].value_counts(), BS['Phase'][BS['Mag']=='FM'].value_counts()], axis=1)

In [ ]:
phase_counts.columns=['NM', 'FM']

In [ ]:
phase_counts.sum().sum()

In [ ]:
phase_counts

In [ ]:
phase_counts.plot.barh(stacked=True)

In [ ]:
fw, fh = plt.rcParams['figure.figsize']

In [ ]:
simple_tcps = pd.Index(['A15', 'C15', 'C14', 'C36', 'chi', 'sigma', 'mu', 'R'])

In [ ]:
simple_tcps.difference(pd.Index(['R']))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# phase_counts is your DataFrame with columns ['NM', 'FM']

# Plot stacked horizontal bars
colors = ['mediumseagreen', 'steelblue'] #['#1f77b4', '#ff7f0e']  # You can choose your own colors

fig, ax = plt.subplots(figsize=(fw, fh/10*len(phase_counts)))
ax.barh(simple_tcps, phase_counts['NM'][simple_tcps], color='mediumseagreen', height=.8)
yticks = ax.get_yticks()
for ( phase, count ), y in zip(phase_counts['NM'][simple_tcps].items(), yticks):
    ax.text(count, y, f'{int(count):d}', ha='right', va='center', color='black', fontsize=14)
ax.barh(simple_tcps.difference(['R']), phase_counts['FM'][simple_tcps.difference(['R'])], left=phase_counts['NM'][simple_tcps.difference(['R'])], color='steelblue', height=.8)
for ( phase, count ), y in zip(phase_counts['FM'][simple_tcps.difference(['R'])].items(), yticks):
    print(phase,  phase)
    ax.text(count+phase_counts['FM'][phase], phase, f'{int(count):d}', ha='right', va='center', color='white', fontsize=14)
#ax.set_xlabel(count)

## Remove samples with very little representation

In [ ]:
BS = BS[BS['Phase'] != 'delta']

# Extra features 

In [ ]:
Features = Featurizer(BS)

In [ ]:
fig, ax = plt.subplots(figsize=(5,8))
sns.histplot(x = Features.Mag, ax=ax)
fig.savefig(f'{dataset}/graphs/{dataset}_FM_NM_counts.eps')

In [ ]:
isfm = Features.Mag.str.contains('FM')

In [ ]:
fmsamples = BS.index[isfm]

In [ ]:
fmsamplesasnm = fmsamples.str.replace('.FM$','.NM')

In [ ]:
nmsamples = BS.index[~isfm]

## nm samples without fm counterpart:

In [ ]:
nmsamples.difference(fmsamplesasnm)

## fm samples as without nm counterpart:

In [ ]:
fmsamplesasnm.difference(nmsamples)

# Distribution of Target Variables 

## total energy

In [ ]:
targets = {'E0':r'$E_0$', 'B0':r'$B_0$', 'V0':r'$V_0$'}

some obvious outliers:

In [ ]:
for target, label in targets.items():
    fig, ax = plt.subplots(figsize = (15,8 ))
    sns.histplot(x = BS[target], ax = ax)
    ax.set_xlabel(label)
    fig.tight_layout()

# General correlation

In [ ]:
BS.sort_values(by='B0', inplace=True)

In [ ]:
targets['E0'] = '$E_0$'

In [ ]:
#fig, ax = plt.subplots()
plt.scatter(BS.V0, BS.B0, c=BS['E0'], marker = 'o' , s = BS.B0/2, cmap='hot', edgecolor='k')
cbar = plt.colorbar()
plt.ylabel(targets['B0']+' (GPa)')
plt.xlabel(targets['V0']+'($\\AA^3$)')
cbar.set_label(targets['E0'])

# Pair Plots

In [ ]:
ToPlot = BS[list(targets.keys())]
ToPlot.columns = list(targets.values()) 

In [ ]:
axis_grid = sns.pairplot(ToPlot, diag_kind = 'kde', kind='reg', height=8)
#, hue='Phase', x_vars=list(targets.values()), y_vars=list(targets.values()), )

# Magnetic vs Non Magnetic

EFFM = BS[target_case][Features.Mag == 'FM']

EFNMfcc = BS[target_case][Features.Mag == 'NM']

EFNMhcp = BS[target_case][Features.Mag == 'NM']

EFFM.index = EFFM.index.str.replace('.FM', '')

EFNMhcp.index = EFNMhcp.index.str.replace('.NM', '')

EFNMhcp.index = EFNMhcp.index.str.replace('.NM', '')

DE_mag  = EFNMhcp - EFFM 

DE_mag[ abs(DE_mag > 0.1)]

DE_mag[DE_mag < 0 ]

ax = sns.histplot(BS[BS.index.str.contains('NM$')]['EF_nmfcc'], color = 'purple', label = "NM fcc ground state")
ax = sns.histplot(BS[BS.index.str.contains('NM$')]['EF_nmhcp'], color='orange', label ='NM hcp ground state')
ax.legend()

x = [-0.1, 0.8] 
y = [-0.1, 0.8]
ax  = sns.scatterplot(EFFM, EFNMhcp, label=target_case)
ax  = sns.scatterplot(EFFM, EFNMfcc, label=target_case, ax=ax, color='purple', size = 20)
ax.plot(x,y, '--k')
ax.set_xlabel(r'$\Delta E _f $ FM  eV/atom')
ax.set_ylabel(r'$\Delta E _f $ NM  eV/atom ')
big_difference = DE_mag[DE_mag.abs()>0.18].index#.index[0]
for bigdifindex in big_difference:
    xy = (EFFM[bigdifindex], EFNMhcp[bigdifindex])
    xytext = (EFFM[bigdifindex],  EFNMhcp[bigdifindex]*1.1)
    ax.annotate(bigdifindex, xy , xytext = xytext, arrowprops={'width': 5, })

BIGDIF_FM = BS.filter(regex='E0|atom_').loc[big_difference+'.FM']

BIGDIF_NM = BS.filter(regex='E0|atom_').loc[big_difference+'.NM']

pd.concat([BIGDIF_FM, BIGDIF_NM], axis = 0)

Features.get_ground_states_energies()

EREF_FM = BS[BS.index.str.contains('FM$', regex=True) & (BS['nelem'] == 1) & (BS['atom_A'] == 'Fe_pv') ][['E0']].sort_values(by='E0').iloc[0]

EREF_FM

BIGDIF_FM['E0'] - EREF_FM['E0']

EREF_NM = BS[BS.index.str.contains('NM$', regex=True) & (BS['nelem'] == 1) & (BS['atom_A'] == 'Fe_pv') ][['E0', 'num_atom_A', 'num_atoms']].sort_values(by='E0').iloc[0]

EREF_NM

BIGDIF_NM['E0'] - EREF_NM['E0']

BS.columns



In [ ]:
BS.query('Phase == "R"')[['Fe_pv','Mo_sv','B0', 'V0','EF_nmhcp']].sort_values(by = 'Mo_sv')